### Explore the structure of folders

In [1]:
%%bash
ls *

LICENSE
README.md
data_exploration.ipynb

Synthea_data:
synthea_sample_data_csv_100_latest
synthea_sample_data_csv_1K_nov2021

UCI_ML_data:
diabetes


In [2]:
%%bash
ls UCI_ML_data/

diabetes


In [3]:
%%bash
file -d UCI_ML_data/diabetes/

UCI_ML_data/diabetes/: directory


In [4]:
%%bash
ls -lsah UCI_ML_data/diabetes/

total 32
 0 drwx------@  6 jiz  staff   192B Dec  5 20:08 .
 0 drwxr-xr-x   4 jiz  staff   128B Dec  5 19:14 ..
16 -rw-r--r--@  1 jiz  staff   6.0K Dec  6 01:44 .DS_Store
 0 drwxr-xr-x@ 75 jiz  staff   2.3K Aug 27  1993 Diabetes-Data
 8 -rwx------@  1 jiz  staff   115B May 22  2023 Index
 8 -rwx------@  1 jiz  staff   682B May 22  2023 README


In [5]:
%%bash
ls -lh UCI_ML_data/diabetes/Diabetes-Data/

total 1600
-rw-r--r--@ 1 jiz  staff   1.7K Aug  6  1993 Data-Codes
-rw-r--r--@ 1 jiz  staff   6.1K Aug 27  1993 Domain-Description
-rw-r--r--@ 1 jiz  staff   2.5K Aug 27  1993 README-DIABETES
-rw-r--r--@ 1 jiz  staff    22K Aug  6  1993 data-01
-rw-r--r--@ 1 jiz  staff    18K Aug  6  1993 data-02
-rw-r--r--@ 1 jiz  staff   6.7K Aug  6  1993 data-03
-rw-r--r--@ 1 jiz  staff   6.7K Aug  6  1993 data-04
-rw-r--r--@ 1 jiz  staff   6.7K Aug  6  1993 data-05
-rw-r--r--@ 1 jiz  staff   3.5K Aug  6  1993 data-06
-rw-r--r--@ 1 jiz  staff   5.7K Aug  6  1993 data-07
-rw-r--r--@ 1 jiz  staff   4.0K Aug  6  1993 data-08
-rw-r--r--@ 1 jiz  staff   4.6K Aug  6  1993 data-09
-rw-r--r--@ 1 jiz  staff   5.8K Aug  6  1993 data-10
-rw-r--r--@ 1 jiz  staff   5.2K Aug  6  1993 data-11
-rw-r--r--@ 1 jiz  staff   6.6K Aug  6  1993 data-12
-rw-r--r--@ 1 jiz  staff   6.6K Aug  6  1993 data-13
-rw-r--r--@ 1 jiz  staff   5.1K Aug  6  1993 data-14
-rw-r--r--@ 1 jiz  staff   6.6K Aug  6  1993 data-15
-rw-r--r--@ 1

### Import the necessary libraries to examine the datasets

In [6]:
import pandas as pd
import numpy as np
import pyampute as imputor
from gensim.models import KeyedVectors as GensimKeyedVecs # unsupervised semantic modeling from plain text
# to explore synonyms, hypernyms, and hyponyms
from nltk.corpus import wordnet as NltkWordNet # linguistically-grounded approach finding the semantic relationships between words

#### Example of using Gensim's KeyedVectors
```python
from gensim.models import KeyedVectors
model = KeyedVectors.load_word2vec_format('model_path', binary=True)
similarity = model.similarity('column_name_one', 'column_name_two')
```

#### Example of using NLTK's wordnet
```python
from nltk.corpus import wordnet as wn
word1 = wn.synsets('column_name_one')[0]  # Assuming the first synset
word2 = wn.synsets('column_name_two')[0]
similarity = word1.wup_similarity(word2)  # Wu-Palmer similarity
```

## Dataset exploration

#### UCI diabetes dataset

In [19]:
%%bash
cat UCI_ML_data/diabetes/Diabetes-Data/README-DIABETES

The DIABETES data sets in this directory are provided for use in 1994 
AI in Medicine symposium submissions.  Permission is granted to use the
data sets for other research purposes as long as appropriate credit is
given as to the source (AIM-94 data set provided by Michael Kahn, MD, PhD, 
Washington University, St. Louis, MO).


Index:
------

* Data-Codes: a listing of the codes used in the data sets.

* Domain-Description: This file describes the basic physiology and patho-
physiology of diabetes mellitus and its treatment.

* data-[01-70]: data sets covering several weeks' to months' worth of
outpatient care on 70 patients.  An additional 10 sets will be made
available two weeks prior to the symposium for interested parties.  Please
contact the organizers if you would like to obtain these data sets.


Methods:
--------

You do not need to use all the data in order to participate.  Use any 
subset of the available data from either the ICU data set or the diabetes 
data set.  Furtherm

In [21]:
%%bash
cat UCI_ML_data/diabetes/Diabetes-Data/Data-Codes

Diabetes patient records were obtained from two sources:  an automatic
electronic recording device and paper records.  The automatic device
had an internal clock to timestamp events, whereas the paper records
only provided "logical time" slots (breakfast, lunch, dinner,
bedtime).  For paper records, fixed times were assigned to breakfast
(08:00), lunch (12:00), dinner (18:00), and bedtime (22:00).  Thus
paper records have fictitious uniform recording times whereas
electronic records have more realistic time stamps.

Diabetes files consist of four fields per record.  Each field is
separated by a tab and each record is separated by a newline.

File Names and format:
(1) Date in MM-DD-YYYY format
(2) Time in XX:YY format
(3) Code
(4) Value

The Code field is deciphered as follows:

33 = Regular insulin dose
34 = NPH insulin dose
35 = UltraLente insulin dose
48 = Unspecified blood glucose measurement
57 = Unspecified blood glucose measurement
58 = Pre-breakfast blood glucose measurement
59

In [22]:
# create the column names from the description above
diabetes_columns = ["Date", "Time", "Code", "Value"]

In [23]:
uci_diabetes_df1 = pd.read_csv("UCI_ML_data/diabetes/Diabetes-Data/data-01", sep="\t", names=diabetes_columns)
uci_diabetes_df2 = pd.read_csv("UCI_ML_data/diabetes/Diabetes-Data/data-02", sep="\t", names=diabetes_columns)

In [25]:
uci_diabetes_df1.head

<bound method NDFrame.head of            Date   Time  Code  Value
0    04-21-1991   9:09    58    100
1    04-21-1991   9:09    33      9
2    04-21-1991   9:09    34     13
3    04-21-1991  17:08    62    119
4    04-21-1991  17:08    33      7
..          ...    ...   ...    ...
938  09-02-1991  17:30    33      7
939  09-02-1991  23:00    48    155
940  09-03-1991   7:20    58    110
941  09-03-1991   7:20    33      9
942  09-03-1991   7:20    34     16

[943 rows x 4 columns]>

In [26]:
uci_diabetes_df2.head

<bound method NDFrame.head of            Date   Time  Code Value
0    10-10-1989  08:00    58   149
1    10-10-1989  08:00    33   010
2    10-10-1989  12:00    60   116
3    10-10-1989  12:00    33   004
4    10-10-1989  18:00    62   304
..          ...    ...   ...   ...
756  01-12-1990  22:00    33   016
757  01-13-1990  08:00    58   272
758  01-13-1990  08:00    33   012
759  01-13-1990  12:00    60   388
760  01-13-1990  12:00    33   006

[761 rows x 4 columns]>

<b>Conclusions</b>: <br>
From the data exmamination above, we could see that the UCI diabetes is a time-series data consisting of four columns only. It cannot be seriously considered as electronic health records (EHR). Therefore, it shall be only considered as a potential benchmark when compared to other existing algorithms. To justify our adoption of federated learning for imputation, we shall use the data source of proper EHRs and then create/construct a trainable and testable pandas dataframe.

<hr/>

#### Synthea dataset